In [ ]:
!pip install opendatasets

import opendatasets as od
import os
import shutil

od.download('https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia')

# Define the root directories
root_dir = '/content/chest-xray-pneumonia/chest_xray'
subdirs = ['train', 'test', 'val']
destination_root = os.path.join(root_dir, 'lung')

# Create the destination root directory if it doesn't exist
os.makedirs(destination_root, exist_ok=True)

# Iterate through each subset directory (train, test, valid)
for subset in subdirs:
    subset_path = os.path.join(root_dir, subset)

    # Iterate through each class folder within the subset
    for class_name in os.listdir(subset_path):
        class_src_path = os.path.join(subset_path, class_name)
        class_dest_path = os.path.join(destination_root, class_name)

        # Create the class folder in the destination if it doesn't exist
        os.makedirs(class_dest_path, exist_ok=True)

        # Move all images from the source class folder to the destination class folder
        for filename in os.listdir(class_src_path):
            src_file = os.path.join(class_src_path, filename)
            dest_file = os.path.join(class_dest_path, filename)

            # Move the file
            shutil.move(src_file, dest_file)

print("Merge complete. All images are now in:", destination_root)


# ============================================================
# FULL REVISED PIPELINE + EDGE ENHANCEMENT BRANCH
#
# Method:
#   - Raw image -> multiscale raw patches
#   - DenseNet201 per crop -> 1920-d node feature
#   - Cache full multiscale graphs
#   - Train baseline GCN on full graphs
#   - Compute perturbation saliency per node
#   - Final model:
#       cross-scale attention
#       collapse to MEDIUM nodes only
#       dense full-image branch -> 8x8 feature grid
#       edge enhancement from dense branch
#       edge-weighted GCN classification
#
# Metrics:
#   - ACC
#   - F1 macro
#   - F1 weighted
#   - Precision
#   - Recall
#   - AUC
# ============================================================

!pip -q install torch-geometric torchvision

import os, glob, hashlib, random
import numpy as np
from PIL import Image
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from torchvision import models, transforms
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

# --------------------------
# Config
# --------------------------
root_dir = "/content/chest-xray-pneumonia/chest_xray/lung"

cache_full_dir = "/content/ms_rawcrop_densenet_graphs"
cache_sal_dir  = "/content/ms_rawcrop_densenet_sal"
os.makedirs(cache_full_dir, exist_ok=True)
os.makedirs(cache_sal_dir, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")
print("Device:", DEVICE)

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

IMG_SIZE = 256

SCALES = [
    {"name": "P128", "patch": 128},
    {"name": "P64",  "patch": 64},
    {"name": "P32",  "patch": 32},
]

CROP_RESIZE = 224
PATCH_ENCODE_BATCH = 84

EPOCHS_BASELINE = 5
BATCH_SIZE_BASELINE = 32
LR_BASE = 1e-3
WEIGHT_DECAY = 1e-4

K_FOLDS = 5
EPOCHS_FINAL = 100
BATCH_SIZE_FINAL = 64
LR_FINAL = 1e-3

NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True if NUM_WORKERS > 0 else False

PERTURB_CHUNK = 32
EDGE_CTX_DIM = 8

# --------------------------
# Helper
# --------------------------
def autocast_context():
    if DEVICE == "cuda":
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return nullcontext()

# --------------------------
# Dataset discovery
# --------------------------
def discover_classes_and_paths(root):
    class_names = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    paths, labels = [], []
    for c in class_names:
        files = []
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
            files += glob.glob(os.path.join(root, c, ext))
        files = sorted(files)
        paths += files
        labels += [class_to_idx[c]] * len(files)

    return class_names, paths, np.array(labels, dtype=np.int64)

class_names, img_paths, labels = discover_classes_and_paths(root_dir)
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Total images:", len(img_paths))
for i, c in enumerate(class_names):
    print(f"  {c}: {int(np.sum(labels == i))}")

# --------------------------
# Cache helpers
# --------------------------
def cache_key(path):
    return hashlib.md5(path.encode("utf-8")).hexdigest()

def full_graph_path(img_path):
    return os.path.join(cache_full_dir, f"{cache_key(img_path)}.pt")

def sal_path(img_path):
    return os.path.join(cache_sal_dir, f"{cache_key(img_path)}.pt")

full_files = [full_graph_path(p) for p in img_paths]
sal_files  = [sal_path(p) for p in img_paths]

# --------------------------
# Transforms
# --------------------------
crop_tf = transforms.Compose([
    transforms.Resize((CROP_RESIZE, CROP_RESIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def load_raw_pil(path):
    img = Image.open(path).convert("L")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    img = img.convert("RGB")
    return img

def load_full_tensor(path):
    img = Image.open(path).convert("L")
    return full_img_tf(img)

# --------------------------
# DenseNet feature extractor
# --------------------------
def make_densenet_encoder():
    m = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
    backbone = m.features
    backbone.eval()
    feat_dim = 1920
    return backbone, feat_dim

backbone, FEAT_DIM = make_densenet_encoder()
backbone = backbone.to(DEVICE)

@torch.no_grad()
def encode_crop_batch(crops_tensor):
    crops_tensor = crops_tensor.to(DEVICE, non_blocking=True)
    with autocast_context():
        fmap = backbone(crops_tensor)
        fmap = F.relu(fmap)
        vec = F.adaptive_avg_pool2d(fmap, 1)
    return vec.view(vec.size(0), -1).float().cpu()

# --------------------------
# Patch metadata
# --------------------------
def build_patch_meta():
    meta_all = []
    for sc in SCALES:
        p = sc["patch"]
        grid = IMG_SIZE // p
        for r in range(grid):
            for c in range(grid):
                x0, y0 = c * p, r * p
                x1, y1 = x0 + p, y0 + p
                cx = (x0 + x1) / 2.0
                cy = (y0 + y1) / 2.0
                meta_all.append({
                    "patch": p,
                    "grid": grid,
                    "row": r,
                    "col": c,
                    "x0": x0, "y0": y0,
                    "x1": x1, "y1": y1,
                    "cx_norm": cx / IMG_SIZE,
                    "cy_norm": cy / IMG_SIZE,
                })
    return meta_all

def build_intrascale_edges(meta_all):
    edges = []
    by_p = {}
    for idx, m in enumerate(meta_all):
        by_p.setdefault(m["patch"], []).append((idx, m))

    for p, items in by_p.items():
        grid = items[0][1]["grid"]
        coord2id = {(m["row"], m["col"]): idx for idx, m in items}
        for idx, m in items:
            r, c = m["row"], m["col"]
            if c + 1 < grid:
                j = coord2id[(r, c + 1)]
                edges += [(idx, j), (j, idx)]
            if r + 1 < grid:
                j = coord2id[(r + 1, c)]
                edges += [(idx, j), (j, idx)]

    return (torch.tensor(edges, dtype=torch.long).t().contiguous()
            if edges else torch.empty((2, 0), dtype=torch.long))

def build_interscale_edges(meta_all):
    patches = [sc["patch"] for sc in SCALES]
    idx_by_p = {p: [] for p in patches}
    for idx, m in enumerate(meta_all):
        idx_by_p[m["patch"]].append((idx, m))

    edges = []
    for li in range(len(patches) - 1):
        p_c, p_f = patches[li], patches[li + 1]
        coarse, fine = idx_by_p[p_c], idx_by_p[p_f]
        if not coarse or not fine:
            continue

        Gc = coarse[0][1]["grid"]
        Gf = fine[0][1]["grid"]
        coord2id_c = {(m["row"], m["col"]): idx for idx, m in coarse}

        for idx_f, m_f in fine:
            r_p = int(np.floor(m_f["row"] * Gc / Gf))
            c_p = int(np.floor(m_f["col"] * Gc / Gf))
            r_p = min(Gc - 1, max(0, r_p))
            c_p = min(Gc - 1, max(0, c_p))
            idx_c = coord2id_c[(r_p, c_p)]
            edges += [(idx_f, idx_c), (idx_c, idx_f)]

    return (torch.tensor(edges, dtype=torch.long).t().contiguous()
            if edges else torch.empty((2, 0), dtype=torch.long))

def crop_patch(pil_img, x0, y0, x1, y1):
    crop = pil_img.crop((x0, y0, x1, y1))
    return crop_tf(crop)

# --------------------------
# Precompute fixed graph structure
# --------------------------
META_TEMPLATE = build_patch_meta()

EDGE_INDEX_TEMPLATE = torch.cat([
    build_intrascale_edges(META_TEMPLATE),
    build_interscale_edges(META_TEMPLATE)
], dim=1).contiguous()

POS_TEMPLATE = torch.tensor(
    [[m["cx_norm"], m["cy_norm"]] for m in META_TEMPLATE],
    dtype=torch.float
)

ROWCOL_TEMPLATE = torch.tensor(
    [[m["row"], m["col"]] for m in META_TEMPLATE],
    dtype=torch.long
)

PATCHSZ_TEMPLATE = torch.tensor(
    [m["patch"] for m in META_TEMPLATE],
    dtype=torch.long
)

BOX_TEMPLATE = torch.tensor(
    [[m["x0"], m["y0"], m["x1"], m["y1"]] for m in META_TEMPLATE],
    dtype=torch.float
)

NODES_PER_GRAPH = len(META_TEMPLATE)
print(f"Nodes per graph: {NODES_PER_GRAPH} (4 coarse + 16 medium + 64 fine)")

COARSE_START = 0
COARSE_END   = 4
MEDIUM_START = 4
MEDIUM_END   = 20
FINE_START   = 20
FINE_END     = 84
MEDIUM_NODES_PER_GRAPH = 16

# --------------------------
# Precompute fixed cross-scale neighbor map
# --------------------------
def build_medium_crossscale_map(meta_all):
    coarse_idx = [i for i, m in enumerate(meta_all) if m["patch"] == 128]
    medium_idx = [i for i, m in enumerate(meta_all) if m["patch"] == 64]
    fine_idx   = [i for i, m in enumerate(meta_all) if m["patch"] == 32]

    boxes = np.array([[m["x0"], m["y0"], m["x1"], m["y1"]] for m in meta_all], dtype=np.float32)
    cx = 0.5 * (boxes[:, 0] + boxes[:, 2])
    cy = 0.5 * (boxes[:, 1] + boxes[:, 3])

    neigh_map = []
    max_len = 0

    for i in medium_idx:
        x0, y0, x1, y1 = boxes[i]
        neigh = []

        for j in coarse_idx:
            if (cx[j] >= x0) and (cx[j] <= x1) and (cy[j] >= y0) and (cy[j] <= y1):
                neigh.append(j)

        for j in fine_idx:
            if (cx[j] >= x0) and (cx[j] <= x1) and (cy[j] >= y0) and (cy[j] <= y1):
                neigh.append(j)

        neigh_map.append(neigh)
        max_len = max(max_len, len(neigh))

    neigh_tensor = torch.full((len(medium_idx), max_len), -1, dtype=torch.long)
    mask_tensor = torch.zeros((len(medium_idx), max_len), dtype=torch.bool)

    for r, neigh in enumerate(neigh_map):
        neigh_tensor[r, :len(neigh)] = torch.tensor(neigh, dtype=torch.long)
        mask_tensor[r, :len(neigh)] = True

    return torch.tensor(medium_idx, dtype=torch.long), neigh_tensor, mask_tensor

MEDIUM_LOCAL_IDX, CSA_NEIGH_IDX, CSA_NEIGH_MASK = build_medium_crossscale_map(META_TEMPLATE)
print("Medium nodes:", MEDIUM_LOCAL_IDX.shape)
print("CSA neighbor index tensor:", CSA_NEIGH_IDX.shape)

# --------------------------
# Medium-grid edge template
# --------------------------
def build_single_medium_edge_template():
    edges = []
    grid = 4

    def nid(r, c):
        return r * grid + c

    for r in range(grid):
        for c in range(grid):
            if c + 1 < grid:
                a, b = nid(r, c), nid(r, c + 1)
                edges += [(a, b), (b, a)]
            if r + 1 < grid:
                a, b = nid(r, c), nid(r + 1, c)
                edges += [(a, b), (b, a)]

    return torch.tensor(edges, dtype=torch.long).t().contiguous()

MEDIUM_EDGE_TEMPLATE = build_single_medium_edge_template()

# --------------------------
# Build full graph
# --------------------------
@torch.no_grad()
def build_full_graph(img_path, label, mini_batch=PATCH_ENCODE_BATCH):
    pil_img = load_raw_pil(img_path)
    crops = [crop_patch(pil_img, m["x0"], m["y0"], m["x1"], m["y1"]) for m in META_TEMPLATE]

    feat_list = []
    for start in range(0, len(crops), mini_batch):
        batch = torch.stack(crops[start:start + mini_batch], dim=0)
        feat_list.append(encode_crop_batch(batch))

    feats = torch.cat(feat_list, dim=0)
    feats = (feats - feats.mean(dim=0, keepdim=True)) / (feats.std(dim=0, keepdim=True) + 1e-6)

    data = Data(
        x=feats,
        edge_index=EDGE_INDEX_TEMPLATE,
        y=torch.tensor([int(label)], dtype=torch.long),
        pos=POS_TEMPLATE
    )
    data.rowcol = ROWCOL_TEMPLATE
    data.patch_size = PATCHSZ_TEMPLATE
    data.box = BOX_TEMPLATE
    return data

# --------------------------
# Cache full graphs
# --------------------------
print("\n[Step 0] Caching FULL graphs (raw crop -> DenseNet per patch)...")
missing = sum(0 if os.path.exists(f) else 1 for f in full_files)
print(f"Missing full graphs: {missing} / {len(full_files)}")

built = 0
for p, y, out in zip(img_paths, labels, full_files):
    if os.path.exists(out):
        continue
    g = build_full_graph(p, y)
    tmp = out + ".tmp"
    torch.save(g, tmp)
    os.replace(tmp, out)
    built += 1
    if built % 50 == 0:
        print(f"  built {built}/{missing}")
print("Full graph caching done.")

# --------------------------
# Baseline model for saliency
# --------------------------
class CachedGraphDataset(Dataset):
    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        return torch.load(self.files[idx], map_location="cpu", weights_only=False)

class BaselineGCN(nn.Module):
    def __init__(self, in_dim, hidden=128, num_classes=2, dropout=0.25):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin1 = nn.Linear(hidden, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, ei))
        g = global_mean_pool(x, batch)
        g = F.relu(self.lin1(g))
        g = F.dropout(g, p=self.dropout, training=self.training)
        return self.lin2(g)

def train_one_epoch_base(model, loader, opt, crit, scaler=None):
    model.train()
    tot_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)

        with autocast_context():
            logits = model(batch)
            y = batch.y.view(-1)
            loss = crit(logits, y)

        if scaler is not None and DEVICE == "cuda":
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()

        tot_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()

    return tot_loss / max(total, 1), correct / max(total, 1)

print("\n" + "=" * 80)
print("[Step 1] Baseline training on FULL graphs")
print("=" * 80)

tmpg = torch.load(full_files[0], map_location="cpu", weights_only=False)
IN_DIM = tmpg.x.size(1)
print("Node feature dim:", IN_DIM)

base_model = BaselineGCN(IN_DIM, hidden=128, num_classes=NUM_CLASSES, dropout=0.25).to(DEVICE)
opt_base = torch.optim.Adam(base_model.parameters(), lr=LR_BASE, weight_decay=WEIGHT_DECAY)
crit = nn.CrossEntropyLoss()
scaler_base = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

full_loader = DataLoader(
    CachedGraphDataset(full_files),
    batch_size=BATCH_SIZE_BASELINE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS
)

for ep in range(1, EPOCHS_BASELINE + 1):
    tr_loss, tr_acc = train_one_epoch_base(base_model, full_loader, opt_base, crit, scaler=scaler_base)
    print(f"Epoch {ep:02d}/{EPOCHS_BASELINE} | train loss {tr_loss:.4f} | train acc {tr_acc:.4f}")

# --------------------------
# Saliency caching
# --------------------------
@torch.no_grad()
def perturb_scores_single_fast(model, data, target_class=None, mask_value=0.0, chunk_size=PERTURB_CHUNK):
    model.eval()

    base_data = data.clone().to(DEVICE)
    base_data.batch = torch.zeros(base_data.num_nodes, dtype=torch.long, device=DEVICE)

    base_probs = F.softmax(model(base_data), dim=1)[0]
    pred_class = int(base_probs.argmax().item())

    if target_class is None:
        target_class = pred_class

    base_score = float(base_probs[target_class].item())
    N = base_data.x.size(0)
    scores = torch.zeros(N, dtype=torch.float32)

    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)

        masked_graphs = []
        for node_idx in range(start, end):
            g = Data(
                x=base_data.x.clone(),
                edge_index=base_data.edge_index,
                y=base_data.y,
                pos=base_data.pos
            )
            g.batch = torch.zeros(g.num_nodes, dtype=torch.long, device=DEVICE)
            g.x[node_idx].fill_(mask_value)
            masked_graphs.append(g)

        batch_obj = Batch.from_data_list(masked_graphs).to(DEVICE)

        with autocast_context():
            probs = F.softmax(model(batch_obj), dim=1)[:, target_class]

        scores[start:end] = (base_score - probs).detach().cpu()

    return scores

print("\n" + "=" * 80)
print("[Step 2] Caching perturbation saliency per node")
print("=" * 80)

missing_sal = sum(0 if os.path.exists(f) else 1 for f in sal_files)
print(f"Missing saliency files: {missing_sal} / {len(sal_files)}")

done = 0
for fin, fout in zip(full_files, sal_files):
    if os.path.exists(fout):
        continue
    g = torch.load(fin, map_location="cpu", weights_only=False)
    s = perturb_scores_single_fast(base_model, g)
    tmp = fout + ".tmp"
    torch.save(s, tmp)
    os.replace(tmp, fout)
    done += 1
    if done % 100 == 0:
        print(f"  scored {done}/{missing_sal}")
print("Saliency caching done.")

# --------------------------
# Full-image Dense branch with Triplet Attention
# --------------------------
class BasicConv(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size,
                              stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class SpatialGate(nn.Module):
    def __init__(self):
        super().__init__()
        self.compress_conv = BasicConv(2, 1, kernel_size=7, stride=1, padding=3)

    def forward(self, x):
        x_max = torch.max(x, dim=1, keepdim=True)[0]
        x_avg = torch.mean(x, dim=1, keepdim=True)
        x_cat = torch.cat([x_max, x_avg], dim=1)
        scale = torch.sigmoid(self.compress_conv(x_cat))
        return x * scale

class TripletAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.cw = SpatialGate()
        self.hc = SpatialGate()
        self.hw = SpatialGate()

    def forward(self, x):
        x_perm1 = x.permute(0, 2, 1, 3).contiguous()
        x_out1 = self.cw(x_perm1)
        x_out1 = x_out1.permute(0, 2, 1, 3).contiguous()

        x_perm2 = x.permute(0, 3, 2, 1).contiguous()
        x_out2 = self.hc(x_perm2)
        x_out2 = x_out2.permute(0, 3, 2, 1).contiguous()

        x_out3 = self.hw(x)
        return (x_out1 + x_out2 + x_out3) / 3.0

class DenseEdgeEnhancer(nn.Module):
    def __init__(self, in_ch=1920, ctx_dim=256):
        super().__init__()
        self.triplet = TripletAttention()
        self.reduce = nn.Conv2d(in_ch, ctx_dim, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(ctx_dim)
        self.act = nn.ReLU(inplace=True)

    def forward(self, fmap):
        fmap = F.adaptive_avg_pool2d(fmap, (8, 8))
        fmap = self.triplet(fmap)
        fmap = self.act(self.bn(self.reduce(fmap)))
        return fmap

# --------------------------
# Cross-scale attention
# --------------------------
class PaperCrossScaleAttentionFast(nn.Module):
    def __init__(self, feat_dim, attn_dim=128, pos_dim=2):
        super().__init__()
        self.Wq = nn.Linear(feat_dim + pos_dim, attn_dim)
        self.Wk = nn.Linear(feat_dim + pos_dim, attn_dim)
        self.Wv = nn.Linear(feat_dim, feat_dim)

    @staticmethod
    def l2norm(t, eps=1e-8):
        return t / (t.norm(p=2, dim=-1, keepdim=True) + eps)

    def forward(self, x, pos, batch_vec, sal):
        device = x.device
        B = int(batch_vec.max().item()) + 1
        N = NODES_PER_GRAPH
        Fdim = x.size(1)

        x_b = x.view(B, N, Fdim)
        pos_b = pos.view(B, N, 2)
        sal_b = sal.view(B, N, 1)

        x_gated = x_b * sal_b
        qk_in = torch.cat([x_gated, pos_b], dim=-1)

        Q = self.l2norm(self.Wq(qk_in))
        K = self.l2norm(self.Wk(qk_in))
        V = self.Wv(x_b)

        medium_idx = MEDIUM_LOCAL_IDX.to(device)
        neigh_idx  = CSA_NEIGH_IDX.to(device)
        neigh_mask = CSA_NEIGH_MASK.to(device)

        Qm = Q[:, medium_idx, :]
        sal_m = sal_b[:, medium_idx, :]

        neigh_idx_safe = neigh_idx.clamp(min=0)
        K_neigh = K[:, neigh_idx_safe, :]
        V_neigh = V[:, neigh_idx_safe, :]
        sal_neigh = sal_b[:, neigh_idx_safe, :]

        sim = (Qm.unsqueeze(2) * K_neigh).sum(dim=-1)
        bias = sal_m.unsqueeze(2) + sal_neigh

        logits = sim.unsqueeze(-1) + bias
        logits = logits.masked_fill(~neigh_mask.unsqueeze(0).unsqueeze(-1), -1e9)

        attn = torch.softmax(logits, dim=2)
        msg = (attn * V_neigh).sum(dim=2)

        x_out = x_b.clone()
        x_out[:, medium_idx, :] = x_out[:, medium_idx, :] + msg

        return x_out.view(B * N, Fdim)

# --------------------------
# Build medium-only graph after CSA
# --------------------------
def build_medium_only_graph_batched_fast(data, x_updated):
    device = x_updated.device
    B = int(data.y.view(-1).numel())
    N = NODES_PER_GRAPH
    Fdim = x_updated.size(1)

    x_b = x_updated.view(B, N, Fdim)
    pos_b = data.pos.view(B, N, 2)

    x_m = x_b[:, MEDIUM_START:MEDIUM_END, :].reshape(B * MEDIUM_NODES_PER_GRAPH, Fdim)
    pos_m = pos_b[:, MEDIUM_START:MEDIUM_END, :].reshape(B * MEDIUM_NODES_PER_GRAPH, 2)

    batch_m = torch.arange(B, device=device).repeat_interleave(MEDIUM_NODES_PER_GRAPH)

    edge_blocks = []
    base_edge = MEDIUM_EDGE_TEMPLATE.to(device)
    for b in range(B):
        edge_blocks.append(base_edge + b * MEDIUM_NODES_PER_GRAPH)
    edge_index = torch.cat(edge_blocks, dim=1)

    out = Data(
        x=x_m,
        edge_index=edge_index,
        y=data.y.view(-1).to(device),
        pos=pos_m
    )
    out.batch = batch_m
    return out

# --------------------------
# Edge enhancement model
# --------------------------
class CSA_MediumOnly_GCN_EdgeEnhance(nn.Module):
    def __init__(self, feat_dim, hidden=128, num_classes=2, dropout=0.25,
                 attn_dim=128, edge_ctx_dim=256):
        super().__init__()
        self.csa = PaperCrossScaleAttentionFast(feat_dim, attn_dim=attn_dim)
        self.edge_branch = DenseEdgeEnhancer(in_ch=1920, ctx_dim=edge_ctx_dim)

        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_ctx_dim * 3, edge_ctx_dim),
            nn.ReLU(inplace=True),
            nn.Linear(edge_ctx_dim, 1)
        )

        self.conv1 = GCNConv(feat_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin1 = nn.Linear(hidden, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def build_edge_weights(self, dense_ctx_map, batch_size, device):
        B, C, H, W = dense_ctx_map.shape
        assert H == 8 and W == 8

        node_ctx = F.avg_pool2d(dense_ctx_map, kernel_size=2, stride=2)   # [B,C,4,4]
        node_ctx = node_ctx.permute(0, 2, 3, 1).contiguous().view(B, 16, C)  # [B,16,C]

        edge_blocks = []
        edge_weight_blocks = []

        base_edge = MEDIUM_EDGE_TEMPLATE.to(device)

        for b in range(B):
            edge_b = base_edge + b * 16
            edge_blocks.append(edge_b)

            src = base_edge[0]
            dst = base_edge[1]

            c_src = node_ctx[b, src, :]
            c_dst = node_ctx[b, dst, :]
            c_diff = torch.abs(c_src - c_dst)

            e_feat = torch.cat([c_src, c_dst, c_diff], dim=1)
            w = torch.sigmoid(self.edge_mlp(e_feat)).view(-1)
            edge_weight_blocks.append(w)

        edge_index = torch.cat(edge_blocks, dim=1)
        edge_weight = torch.cat(edge_weight_blocks, dim=0)
        return edge_index, edge_weight

    def forward(self, data, sal, full_img):
        x_updated = self.csa(
            x=data.x,
            pos=data.pos,
            batch_vec=data.batch,
            sal=sal
        )

        g = build_medium_only_graph_batched_fast(data, x_updated)

        B = int(data.y.view(-1).numel())

        # Fix PyG batching for full_img:
        # [B*3, H, W] -> [B, 3, H, W]
        if full_img.dim() == 3:
            full_img = full_img.view(B, 3, IMG_SIZE, IMG_SIZE)
        elif full_img.dim() == 4 and full_img.size(1) != 3:
            full_img = full_img.view(B, 3, IMG_SIZE, IMG_SIZE)

        with autocast_context():
            fmap = backbone(full_img)
            fmap = F.relu(fmap)

        dense_ctx_map = self.edge_branch(fmap)

        edge_index, edge_weight = self.build_edge_weights(dense_ctx_map, B, g.x.device)

        x2 = F.relu(self.conv1(g.x, edge_index, edge_weight=edge_weight))
        x2 = F.dropout(x2, p=self.dropout, training=self.training)
        x2 = F.relu(self.conv2(x2, edge_index, edge_weight=edge_weight))

        pooled = global_mean_pool(x2, g.batch)
        pooled = F.relu(self.lin1(pooled))
        pooled = F.dropout(pooled, p=self.dropout, training=self.training)
        return self.lin2(pooled)

# --------------------------
# Dataset with graph + saliency + full image
# --------------------------
class GraphWithSalAndImageDataset(Dataset):
    def __init__(self, graph_files, sal_files, img_paths):
        self.graph_files = graph_files
        self.sal_files   = sal_files
        self.img_paths   = img_paths

    def __len__(self):
        return len(self.graph_files)

    def __getitem__(self, idx):
        g = torch.load(self.graph_files[idx], map_location="cpu", weights_only=False)
        s = torch.load(self.sal_files[idx], map_location="cpu", weights_only=False)
        img = load_full_tensor(self.img_paths[idx])
        g.node_sal = s.float()
        g.full_img = img
        return g

# --------------------------
# Train / eval helpers
# --------------------------
def train_one_epoch_edge(model, loader, opt, crit, scaler=None):
    model.train()
    tot_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        sal = batch.node_sal.to(DEVICE, non_blocking=True)
        full_img = batch.full_img.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with autocast_context():
            logits = model(batch, sal, full_img)
            y = batch.y.view(-1).to(DEVICE)
            loss = crit(logits, y)

        if scaler is not None and DEVICE == "cuda":
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()

        tot_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()

    return tot_loss / max(total, 1), correct / max(total, 1)

@torch.no_grad()
def eval_edge(model, loader, num_classes):
    model.eval()

    ys, preds, probs_all = [], [], []
    correct, total = 0, 0

    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        sal = batch.node_sal.to(DEVICE, non_blocking=True)
        full_img = batch.full_img.to(DEVICE, non_blocking=True)

        with autocast_context():
            logits = model(batch, sal, full_img)

        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(1)
        y = batch.y.view(-1).to(DEVICE)

        correct += (pred == y).sum().item()
        total += y.numel()

        ys.append(y.detach().cpu().numpy())
        preds.append(pred.detach().cpu().numpy())
        probs_all.append(probs.detach().cpu().numpy())

    acc = correct / max(total, 1)
    y_true = np.concatenate(ys) if ys else np.array([])
    y_pred = np.concatenate(preds) if preds else np.array([])
    y_prob = np.concatenate(probs_all, axis=0) if probs_all else np.array([])

    if y_true.size == 0:
        return {"acc":0.0,"f1_macro":0.0,"f1_weighted":0.0,"precision":0.0,"recall":0.0,"auc":0.0}

    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)

    try:
        if num_classes == 2:
            auc = roc_auc_score(y_true, y_prob[:, 1])
        else:
            auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")

    return {
        "acc": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision": precision,
        "recall": recall,
        "auc": auc
    }

# ============================================================
# Step 4: 5-fold CV with edge enhancement branch
# ============================================================
print("\n" + "=" * 90)
print("[Step 4] 5-fold CV: CSA -> medium graph -> dense branch edge enhancement -> GCN")
print("=" * 90)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_best_metrics = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), start=1):
    print("\n" + "=" * 90)
    print(f"FOLD {fold}/{K_FOLDS}  (EDGE ENHANCEMENT BRANCH)")
    print("=" * 90)

    tr_g = [full_files[i] for i in tr_idx]
    tr_s = [sal_files[i]  for i in tr_idx]
    tr_p = [img_paths[i]  for i in tr_idx]

    va_g = [full_files[i] for i in va_idx]
    va_s = [sal_files[i]  for i in va_idx]
    va_p = [img_paths[i]  for i in va_idx]

    print("Loading fold data into memory...")
    train_dataset = GraphWithSalAndImageDataset(tr_g, tr_s, tr_p)
    val_dataset   = GraphWithSalAndImageDataset(va_g, va_s, va_p)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE_FINAL,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE_FINAL,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS
    )

    sanity = next(iter(train_loader))
    B_sanity = int(sanity.y.view(-1).numel())
    print("Sanity | graphs in batch:", B_sanity,
          "| total nodes:", int(sanity.num_nodes),
          "| medium nodes:", int((sanity.patch_size == 64).sum().item()),
          "| raw full_img batch:", tuple(sanity.full_img.shape),
          "| reshaped full_img batch:", (B_sanity, 3, IMG_SIZE, IMG_SIZE))

    model = CSA_MediumOnly_GCN_EdgeEnhance(
        feat_dim=IN_DIM,
        hidden=128,
        num_classes=NUM_CLASSES,
        dropout=0.25,
        attn_dim=128,
        edge_ctx_dim=EDGE_CTX_DIM
    ).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=LR_FINAL, weight_decay=WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()
    scaler_final = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    best_acc = -1.0
    best_metrics = None

    for ep in range(1, EPOCHS_FINAL + 1):
        tr_loss, tr_acc = train_one_epoch_edge(model, train_loader, opt, crit, scaler=scaler_final)
        val_metrics = eval_edge(model, val_loader, NUM_CLASSES)

        print(
            f"Epoch {ep:03d}/{EPOCHS_FINAL} | "
            f"train loss {tr_loss:.4f} | train acc {tr_acc:.4f} | "
            f"val acc {val_metrics['acc']:.4f} | "
            f"val f1_macro {val_metrics['f1_macro']:.4f} | "
            f"val f1_weighted {val_metrics['f1_weighted']:.4f} | "
            f"val prec {val_metrics['precision']:.4f} | "
            f"val rec {val_metrics['recall']:.4f} | "
            f"val auc {val_metrics['auc']:.4f}"
        )

        if val_metrics["acc"] > best_acc:
            best_acc = val_metrics["acc"]
            best_metrics = val_metrics.copy()

    fold_best_metrics.append(best_metrics)

    print(
        f"-> Best fold {fold}: "
        f"ACC {best_metrics['acc']:.4f} | "
        f"F1-macro {best_metrics['f1_macro']:.4f} | "
        f"F1-weighted {best_metrics['f1_weighted']:.4f} | "
        f"Precision {best_metrics['precision']:.4f} | "
        f"Recall {best_metrics['recall']:.4f} | "
        f"AUC {best_metrics['auc']:.4f}"
    )

print("\n" + "=" * 100)
print("EDGE ENHANCEMENT BRANCH | 5-FOLD SUMMARY")
print("=" * 100)

for i, m in enumerate(fold_best_metrics, start=1):
    print(
        f"Fold {i}: "
        f"ACC {m['acc']:.4f} | "
        f"F1-macro {m['f1_macro']:.4f} | "
        f"F1-weighted {m['f1_weighted']:.4f} | "
        f"Precision {m['precision']:.4f} | "
        f"Recall {m['recall']:.4f} | "
        f"AUC {m['auc']:.4f}"
    )

accs = np.array([m["acc"] for m in fold_best_metrics], dtype=float)
f1_macros = np.array([m["f1_macro"] for m in fold_best_metrics], dtype=float)
f1_weighteds = np.array([m["f1_weighted"] for m in fold_best_metrics], dtype=float)
precs = np.array([m["precision"] for m in fold_best_metrics], dtype=float)
recs = np.array([m["recall"] for m in fold_best_metrics], dtype=float)
aucs = np.array([m["auc"] for m in fold_best_metrics], dtype=float)

print("-" * 100)
print(f"Mean ACC         : {np.nanmean(accs):.4f} ± {np.nanstd(accs):.4f}")
print(f"Mean F1-macro    : {np.nanmean(f1_macros):.4f} ± {np.nanstd(f1_macros):.4f}")
print(f"Mean F1-weighted : {np.nanmean(f1_weighteds):.4f} ± {np.nanstd(f1_weighteds):.4f}")
print(f"Mean Precision   : {np.nanmean(precs):.4f} ± {np.nanstd(precs):.4f}")
print(f"Mean Recall      : {np.nanmean(recs):.4f} ± {np.nanstd(recs):.4f}")
print(f"Mean AUC         : {np.nanmean(aucs):.4f} ± {np.nanstd(aucs):.4f}")
print("=" * 100)
